# 📊 Notebook 4 — Full Visualizations & Key Insights

**Goal:** Generate all publication-ready charts and summarize findings.

---
Charts generated:
1. Monthly Revenue Trend  
2. Top 10 Products by Revenue  
3. Top 10 Countries by Revenue  
4. RFM Segment Distribution  
5. Pareto 80/20 Analysis  
6. Revenue Distribution (boxplot)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../outputs', exist_ok=True)

PALETTE = {'primary': '#2563EB', 'secondary': '#F59E0B', 'success': '#10B981',
           'danger': '#EF4444', 'neutral': '#6B7280', 'bg': '#F8FAFC'}
SEGMENT_COLORS = {'Champion': '#7C3AED', 'Loyal': '#2563EB', 'At Risk': '#F59E0B', 'Lost': '#EF4444'}

plt.rcParams.update({'font.family': 'DejaVu Sans', 'figure.facecolor': PALETTE['bg'],
                     'axes.facecolor': PALETTE['bg'], 'axes.spines.top': False, 'axes.spines.right': False})
print('Setup complete ✅')

In [ ]:
# Load all datasets
rfm      = pd.read_csv('../data/rfm_analysis.csv')
segments = pd.read_csv('../data/customer_segments.csv')
pareto   = pd.read_csv('../data/pareto_analysis.csv')

print(f'RFM: {len(rfm):,} customers')
print(f'Segments: {len(segments):,} customers')
print(f'Pareto: {len(pareto):,} customers')

## 📈 Chart 1 — Pareto 80/20 Analysis

In [ ]:
top200 = pareto.head(200).reset_index(drop=True)
idx_80 = (pareto['RevenuePct'] >= 80).idxmax()

fig, ax1 = plt.subplots(figsize=(14, 6))
ax2 = ax1.twinx()

ax1.bar(top200.index, top200['Monetary'], color=PALETTE['primary'], alpha=0.6, label='Revenue per Customer')
ax2.plot(top200.index, top200['RevenuePct'], color=PALETTE['danger'], linewidth=2.5, label='Cumulative Revenue %')

ax2.axhline(80, color=PALETTE['secondary'], linestyle='--', linewidth=1.5, label='80% threshold')
cap_x = min(idx_80, 199)
ax2.axvline(cap_x, color=PALETTE['success'], linestyle='--', linewidth=1.5,
            label=f'{idx_80+1:,} customers = 80% of revenue')

ax1.set_xlabel('Customer Rank (by Revenue)', fontsize=11)
ax1.set_ylabel('Individual Revenue (£)', fontsize=11)
ax2.set_ylabel('Cumulative Revenue (%)', fontsize=11)
ax2.set_ylim(0, 105)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'£{v/1e3:.0f}K'))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=9)

ax1.set_title('Pareto Analysis — 80/20 Rule: Top Customers Drive Revenue', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/pareto_chart.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n💡 INSIGHT: Just {idx_80+1:,} of {len(pareto):,} customers ({(idx_80+1)/len(pareto)*100:.1f}%) generate 80% of total revenue.')

## 👥 Chart 2 — RFM Segment Distribution

In [ ]:
counts = segments['Segment'].value_counts()
seg_colors = [SEGMENT_COLORS.get(s, PALETTE['neutral']) for s in counts.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=PALETTE['bg'])

axes[0].bar(counts.index, counts.values, color=seg_colors, edgecolor='white', width=0.6)
for i, (seg, cnt) in enumerate(counts.items()):
    axes[0].text(i, cnt + 30, f'{cnt:,}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Customer Count by Segment', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Customers', fontsize=11)

wedges, texts, autotexts = axes[1].pie(
    counts.values, labels=counts.index, colors=seg_colors,
    autopct='%1.1f%%', startangle=140, pctdistance=0.75,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
for at in autotexts: at.set_fontsize(10); at.set_fontweight('bold')
axes[1].set_title('Segment Share (%)', fontsize=13, fontweight='bold')

fig.suptitle('RFM Customer Segmentation Overview', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/rfm_segment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 📦 Chart 3 — Revenue Distribution by Segment

In [ ]:
seg_rev = segments.groupby('Segment')['Monetary'].sum().sort_values(ascending=False)
seg_colors2 = [SEGMENT_COLORS.get(s, PALETTE['neutral']) for s in seg_rev.index]

fig, ax = plt.subplots(figsize=(9, 5), facecolor=PALETTE['bg'])
bars = ax.bar(seg_rev.index, seg_rev.values, color=seg_colors2, edgecolor='white', width=0.5)
for bar, val in zip(bars, seg_rev.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50000,
            f'£{val/1e6:.1f}M', ha='center', fontsize=11, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'£{v/1e6:.0f}M'))
ax.set_title('Total Revenue Contributed by Each Segment', fontsize=13, fontweight='bold')
ax.set_ylabel('Total Revenue', fontsize=11)
plt.tight_layout()
plt.savefig('../outputs/revenue_by_segment.png', dpi=150, bbox_inches='tight')
plt.show()

## 💡 Key Insights Summary

In [ ]:
total_rev = segments['Monetary'].sum()
loyal_rev = segments[segments['Segment']=='Loyal']['Monetary'].sum()
at_risk_rev = segments[segments['Segment']=='At Risk']['Monetary'].sum()
pareto_80_pct = (pareto['RevenuePct'] >= 80).idxmax() + 1

print('=' * 55)
print('  📊 KEY BUSINESS INSIGHTS')
print('=' * 55)
print(f'  Total Revenue:         £{total_rev:>12,.2f}')
print(f'  Loyal Customer Revenue:£{loyal_rev:>12,.2f}  ({loyal_rev/total_rev*100:.1f}%)')
print(f'  At-Risk Revenue:       £{at_risk_rev:>12,.2f}  ({at_risk_rev/total_rev*100:.1f}%)')
print(f'  Pareto (80% revenue):  {pareto_80_pct:>6,} of {len(pareto):,} customers ({pareto_80_pct/len(pareto)*100:.1f}%)')
print('=' * 55)
print()
print('  💡 RECOMMENDATIONS')
print('  1. Launch VIP programme for 920 Loyal customers')
print('  2. Win-back campaign targeting At Risk customers')
print('  3. Focus on top 1,354 customers for 80% of ROI')
print('  4. UK dominates — explore EU market growth')
print('=' * 55)